In [9]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.metrics import mean_absolute_error
import numpy as np

In [2]:
df = pd.read_csv("./data/df_final_janeiro_18.csv")
df

,Autoavaliacao_Saude,Idade,Sexo,Cor_Raca,Estado_Civil,Nivel_Instrucao,Total_Moradores,Qtd_Comodos,Qtd_Banheiros,Possui_Automovel,...,Diagnostico_Doenca_Cronica,Grau_Limite_Depressao,Grau_Limite_Doenca_Mental,Grau_Limite_DORT,Pratica_Exercicio,Horas_TV,Fuma,Freq_Alcool,Apoio_Familiar,Apoio_Amigos
0,3.0,55.0,2.0,1.0,1.0,1.0,6.0,5.0,2.0,2.0,...,1.0,2.0,2.0,1.0,1.0,3.0,3.0,2.0,3.0,0.0
1,2.0,19.0,2.0,4.0,4.0,1.0,4.0,4.0,1.0,2.0,...,2.0,2.0,2.0,2.0,2.0,6.0,1.0,2.0,3.0,2.0
2,3.0,45.0,2.0,2.0,4.0,1.0,8.0,8.0,4.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,3.0,3.0,1.0,1.0
3,3.0,58.0,2.0,2.0,3.0,1.0,1.0,1.0,0.0,2.0,...,2.0,2.0,2.0,2.0,1.0,1.0,3.0,1.0,1.0,0.0
4,2.0,28.0,2.0,4.0,4.0,2.0,2.0,2.0,1.0,2.0,...,2.0,2.0,2.0,2.0,1.0,2.0,3.0,1.0,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90841,3.0,54.0,2.0,4.0,1.0,1.0,3.0,5.0,1.0,1.0,...,1.0,2.0,2.0,2.0,2.0,3.0,1.0,3.0,3.0,3.0
90842,2.0,44.0,1.0,4.0,2.0,1.0,2.0,5.0,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,6.0,1.0,3.0,2.0,2.0
90843,2.0,32.0,2.0,4.0,1.0,2.0,4.0,4.0,1.0,2.0,...,2.0,2.0,2.0,2.0,1.0,4.0,3.0,1.0,3.0,0.0
90844,3.0,54.0,1.0,4.0,1.0,1.0,3.0,10.0,3.0,1.0,...,1.0,1.0,2.0,1.0,1.0,2.0,3.0,1.0,1.0,3.0


In [3]:
target = "Autoavaliacao_Saude"

In [6]:
# Splitando a variavel em X e Y
y = df[target]
X = df.drop(columns=[target])

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)


model = OrderedModel(
    y,
    X_scaled,
    distr="logit" # "probit"
)

result = model.fit(method="bfgs")

print(result.summary())

Optimization terminated successfully.
         Current function value: 1.015671
         Iterations: 26
         Function evaluations: 27
         Gradient evaluations: 27
                              OrderedModel Results                             
Dep. Variable:     Autoavaliacao_Saude   Log-Likelihood:                -92270.
Model:                    OrderedModel   AIC:                         1.846e+05
Method:             Maximum Likelihood   BIC:                         1.849e+05
Date:                 Wed, 21 Jan 2026                                         
Time:                         22:54:36                                         
No. Observations:                90846                                         
Df Residuals:                    90817                                         
Df Model:                           25                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
-------------

In [8]:
coef = pd.DataFrame({
    "coef": result.params[:len(X.columns)],
    "p_value": result.pvalues[:len(X.columns)]
}, index=X.columns)

coef["odds_ratio"] = coef["coef"].apply(lambda x: np.exp(x))
coef = coef.sort_values("coef")
coef

,coef,p_value,odds_ratio
Diagnostico_Doenca_Cronica,-0.446226,0.000000e+00,0.640039
Freq_Alcool,-0.158757,8.229106e-108,0.853204
Qtd_Banheiros,-0.132611,8.238793e-43,0.875805
Sexo,-0.097605,5.627869e-43,0.907007
Apoio_Amigos,-0.089005,6.605802e-37,0.914841
Qtd_Comodos,-0.046656,1.198319e-06,0.954416
Grau_Limite_Doenca_Mental,-0.035562,6.690445e-08,0.965063
Apoio_Familiar,-0.030426,1.277183e-05,0.970032
Fuma,-0.019885,3.396751e-03,0.980312
Estado_Civil,-0.016962,1.808188e-02,0.983181


Regra de interpretação

Coeficiente > 0 → aumenta a chance de a pessoa ir para categorias PIORES de saúde

Coeficiente < 0 → aumenta a chance de ir para categorias MELHORES de saúde